# Olist SQL Validation

This notebook checks the main SQL results against the cleaned PostgreSQL data. It focuses on simple totals, row counts, missing values, and the delivery and review metrics used in the project.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PGUSER', 'postgres')}:{os.getenv('PGPASSWORD', '')}"
    f"@{os.getenv('PGHOST', 'localhost')}:{os.getenv('PGPORT', '5432')}"
    f"/{os.getenv('PGDATABASE', 'olist_database')}"
)
print('Database connection ready')

Database connection ready


## Basic table checks

In [2]:
table_checks = pd.read_sql("""
SELECT
    COUNT(*) AS order_rows,
    COUNT(*) FILTER (WHERE order_id IS NULL) AS missing_order_ids,
    COUNT(*) FILTER (WHERE order_purchase_timestamp IS NULL) AS missing_purchase_dates
FROM cleaned.olist_orders_dataset
""", engine)
table_checks

,order_rows,missing_order_ids,missing_purchase_dates
0,99441,0,0


## Monthly sales totals

In [8]:
monthly_sales = pd.read_sql("""
SELECT
    DATE_TRUNC('month', orders.order_purchase_timestamp)::date AS sales_month,
    COUNT(DISTINCT orders.order_id) AS delivered_orders,
    SUM(items.price) AS product_revenue,
    SUM(items.freight_value) AS freight_value,
    SUM(items.price + items.freight_value) AS total_order_value
FROM cleaned.olist_orders_dataset AS orders
INNER JOIN cleaned.olist_order_items_dataset AS items
    ON orders.order_id = items.order_id
WHERE orders.order_status = 'delivered'
GROUP BY 1
ORDER BY total_order_value DESC
""", engine)
print(f'Months returned: {len(monthly_sales)}')
print(f'Total customer order value: {monthly_sales.total_order_value.sum():,.2f}')
monthly_sales

Months returned: 23
Total customer order value: 15,419,773.75


,sales_month,delivered_orders,product_revenue,freight_value,total_order_value
0,2017-11-01,7289,987765.37,165598.83,1153364.20
1,2018-04-01,6798,973534.09,159344.84,1132878.93
2,2018-05-01,6749,977544.69,151229.83,1128774.52
3,2018-03-01,7003,953356.25,167241.99,1120598.24
4,2018-01-01,7069,924645.00,153242.46,1077887.46
5,2018-07-01,6159,867953.46,159853.82,1027807.28
6,2018-06-01,6099,856077.86,155900.43,1011978.29
7,2018-08-01,6351,838576.64,146915.00,985491.64
8,2018-02-01,6555,826437.13,139731.28,966168.41
9,2017-12-01,5513,726033.19,117045.10,843078.29


## Delivery and review checks

In [4]:
delivery_reviews = pd.read_sql("""
WITH order_reviews AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM cleaned.olist_order_reviews_dataset
    WHERE review_score IS NOT NULL
    GROUP BY order_id
), delivered AS (
    SELECT
        CASE
            WHEN order_delivered_customer_date::date < order_estimated_delivery_date::date THEN 'early'
            WHEN order_delivered_customer_date::date = order_estimated_delivery_date::date THEN 'on_time'
            ELSE 'late'
        END AS delivery_performance,
        review_score
    FROM cleaned.olist_orders_dataset AS orders
    INNER JOIN order_reviews USING (order_id)
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)
SELECT delivery_performance, COUNT(*) AS orders, ROUND(AVG(review_score)::numeric, 2) AS average_review_score
FROM delivered
GROUP BY delivery_performance
ORDER BY delivery_performance
""", engine)
delivery_reviews

,delivery_performance,orders,average_review_score
0,early,88163,4.29
1,late,6381,2.27
2,on_time,1280,4.04
